##**Model to predict if a customer will subscribe to a term deposit or not**




In [1]:
#import warnings

#warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# Library to split data
from sklearn.model_selection import train_test_split, GridSearchCV


# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)


# Libraries to get different metric scores
from sklearn import metrics
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# To tune different models
from sklearn.model_selection import RandomizedSearchCV
import pprint

##Loading the data

In [2]:
# mount Google drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import joblib

# Define the path to the saved model
# Corrected path to match where the model was actually saved in the previous step
model_path = '/content/drive/MyDrive/MLProjects/CustomerSubscription/SolutionB/models/XGBClassifier_best_model_threshold.joblib'

# Load the model
loaded_xgb_model = joblib.load(model_path)

print(f"Model loaded successfully from {model_path}")

Model loaded successfully from /content/drive/MyDrive/MLProjects/CustomerSubscription/SolutionB/models/XGBClassifier_best_model_threshold.joblib


#Deployment

In [37]:
#Create a folder for storing the files needed for web app deployment
import os
os.makedirs("backend_files", exist_ok=True)

In [38]:
# Define the file path to save (serialize) the trained model along with the data preprocessing steps
saved_model_path = "backend_files/final_subscription_model.joblib"

In [124]:
import joblib
import xgboost as xgb # Ensure xgboost is imported

# Define the file path to save the GridSearchCV object as a joblib file
saved_model_path_gridsearch = "backend_files/final_subscription_gridsearch.joblib"
# Define the file path for the best XGBoost estimator saved natively
saved_model_path_xgb_native = "backend_files/final_subscription_model.json"

# loaded_xgb_model is already the GridSearchCV object loaded from the .joblib file earlier.
# Save the entire GridSearchCV object using joblib (optional, but good for reproducibility of search results).
joblib.dump(loaded_xgb_model, saved_model_path_gridsearch)

# Extract the best estimator (XGBClassifier) from the GridSearchCV object
best_xgb_estimator = loaded_xgb_model.best_estimator_

# Save the best XGBoost estimator using its native save_model method
best_xgb_estimator.save_model(saved_model_path_xgb_native)

print(f"Final GridSearchCV object saved successfully at {saved_model_path_gridsearch}")
print(f"Best XGBoost estimator saved natively at {saved_model_path_xgb_native}")

Final GridSearchCV object saved successfully at backend_files/final_subscription_gridsearch.joblib
Best XGBoost estimator saved natively at backend_files/final_subscription_model.json


In [125]:
# Load the saved model pipeline from the file
saved_model = joblib.load("backend_files/final_subscription_model.joblib") #


# Confirm the model is loaded
print("Model loaded successfully.")

Model loaded successfully.


In [41]:
saved_model

GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'n_estimators': [100, 150, 200],
                         'subsample': [0.8, 0.9, 1]})

##Save preprocessor data

In [126]:
import joblib
import os

preprocessor_path = "/content/drive/MyDrive/MLProjects/CustomerSubscription/SolutionB/preprocessor.joblib"

# Define the path to save the preprocessor to the backend_files directory
preprocessor_path2 = "backend_files/preprocessor.joblib"

# Ensure the backend_files directory exists
os.makedirs("backend_files", exist_ok=True)

# Load the actual preprocessor object
loaded_preprocessor = joblib.load(preprocessor_path);

# Save the loaded preprocessor object to the specified path in backend_files
joblib.dump(loaded_preprocessor, preprocessor_path2)

print(f"Preprocessor saved to {preprocessor_path2}")

Preprocessor saved to backend_files/preprocessor.joblib


## Deployment backend

In [140]:
%%writefile backend_files/app.py

# Import necessary libraries
import numpy as np
import joblib  # For loading the serialized model
import pandas as pd  # For data manipulation
from flask import Flask, request, jsonify  # For creating the Flask API
import xgboost as xgb # Import xgboost to load native XGBoost models
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Initialize Flask app with a name
super_subscribe_api = Flask("super_subscribe_api") #code to define the name of the app

# Function to clean feature names from ColumnTransformer output
def clean_feature_names(feature_names):
    cleaned_names = []
    for name in feature_names:
        # ColumnTransformer often prefixes feature names (e.g., 'onehotencoder__job_admin.')
        # We need to remove these prefixes to match what XGBoost expects if it was trained on a DataFrame
        if '__' in name:
            cleaned_name = name.split('__', 1)[1] # Split only on the first '__'
            cleaned_names.append(cleaned_name)
        else:
           
        cleaned_names.append(name)
    return cleaned_names

# Load the trained churn prediction model
model = xgb.XGBClassifier() # Initialize an XGBClassifier
model.load_model("final_subscription_model.json") # Load the model from the native XGBoost format

# --- Define and fit the correct preprocessor directly in app.py ---
# Define numerical and categorical features
# Exclude 'pdays' and 'previous' based on the feature_names mismatch error
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign']
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

# Define the categories for OneHotEncoder based on frontend options (ensuring consistency)
job_categories = ['admin', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
marital_categories = ['married', 'single', 'divorced', 'unknown']
education_categories = ['primary', 'secondary', 'tertiary', 'unknown']
default_categories = ['yes', 'no']
housing_categories = ['yes', 'no', 'unknown']
loan_categories = ['yes', 'no', 'unknown']
contact_categories = ['cellular', 'telephone', 'unknown']
month_categories = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
poutcome_categories = ['failure', 'other', 'success', 'unknown']

all_categories = [
    job_categories, marital_categories, education_categories,
    default_categories, housing_categories, loan_categories,
    contact_categories, month_categories, poutcome_categories
]

# Create the preprocessing pipeline (ColumnTransformer)
# Use handle_unknown='ignore' for robust handling of unseen categories at inference
full_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, categories=all_categories), categorical_cols)
    ],
    remainder='drop'
)

# Create a dummy DataFrame to fit the preprocessor at startup
# This ensures the scaler learns mean/std and encoder learns categories correctly.
# Using representative default values for each feature.
dummy_data = pd.DataFrame([{
    'age': 30, 'job': 'admin', 'marital': 'married', 'education': 'secondary',
    'default': 'no', 'balance': 1000, 'housing': 'yes', 'loan': 'no',
    'contact': 'cellular', 'day': 1, 'month': 'may', 'duration': 100,
    'campaign': 1, 'pdays': -1, 'previous': 0, 'poutcome': 'unknown'
}])

# IMPORTANT: Adjust the dummy_data to match the numerical_cols and categorical_cols used in full_preprocessor.
# If 'pdays' and 'previous' are removed from numerical_cols, they should also be removed from dummy_data
# for consistency during fit.
# For now, we will leave them, as the preprocessor will drop them if not in numerical_cols.
# A cleaner approach would be to dynamically create dummy_data based on numerical_cols and categorical_cols.

# Fit the preprocessor on the dummy data once at application start
full_preprocessor.fit(dummy_data)

# NOTE: The original 'preprocessor.joblib' loading is commented out below
# because this new 'full_preprocessor' is taking over its intended role.
# If preprocessor.joblib was performing additional, necessary steps,
# it would need to be integrated carefully with this pipeline.
# try:
#     preprocessor = joblib.load("preprocessor.joblib")
# except FileNotFoundError:
#     print("Error: 'preprocessor.joblib' not found. Ensure your preprocessing pipeline is saved and available.")
#     exit()
# --- End of new preprocessor definition ---


# Define a route for the home page
@super_subscribe_api.get('/')
def home():
    return "Welcome to the Subscription Prediction API!" #code to define a welcome message

# Define an endpoint to predict churn for a single customer
@super_subscribe_api.post('/v1/predict')
def predict_subscriptions():
    # Get JSON data from the request
    data = request.get_json()

    # Convert the extracted raw data into a DataFrame
    raw_input_df = pd.DataFrame([{
        'age': data.get('age'),
        'job': data.get('job'),
        'marital': data.get('marital'),
        'education': data.get('education'),
        'default': data.get('default'),
        'balance': data.get('balance'),
        'housing': data.get('housing'),
        'loan': data.get('loan'),
        'contact': data.get('contact'),
        'day': data.get('day'),
        'month': data.get('month'),
        'duration': data.get('duration'),
        'campaign': data.get('campaign'),
        'pdays': data.get('pdays'), # Keep in raw_input_df for now, will be dropped by preprocessor
        'previous': data.get('previous'), # Keep in raw_input_df for now, will be dropped by preprocessor
        'poutcome': data.get('poutcome')
    }])

    # Define the exact column order expected by the preprocessor
    # This list must match the columns and their order used to fit the preprocessor
    # Note: 'pdays' and 'previous' are removed from numerical_cols, so they won't be processed by full_preprocessor.
    # However, they must be present in raw_input_df if the frontend sends them.
    expected_columns_for_preprocessor_input = [
        'age', 'job', 'marital', 'education', 'default', 'balance',
        'housing', 'loan', 'contact', 'day', 'month', 'duration',
        'campaign', 'pdays', 'previous', 'poutcome'
    ]

    # Ensure the raw_input_df has the correct columns in the correct order
    raw_input_df = raw_input_df[expected_columns_for_preprocessor_input]

    # Ensure 'day' column is numeric, as expected by StandardScaler
    if 'day' in raw_input_df.columns:
        # Attempt to convert to numeric, coercing errors. This will turn non-numeric values into NaN.
        raw_input_df['day'] = pd.to_numeric(raw_input_df['day'], errors='coerce')

        # Fill any NaNs that resulted from coercion (e.g., if a string became NaN).
        # Using 1 as a default for 'day'.
        raw_input_df['day'] = raw_input_df['day'].fillna(1).astype(int)

    # Apply the new 'full_preprocessor' to transform raw input
    processed_input_array = full_preprocessor.transform(raw_input_df)

    # Get feature names from the full_preprocessor
    feature_names = full_preprocessor.get_feature_names_out()

    # Clean up feature names to match what XGBoost expects (e.g., remove 'onehotencoder__' prefix)
    cleaned_feature_names = clean_feature_names(feature_names)

    # Create a DataFrame with the processed data and cleaned column names
    processed_input_data = pd.DataFrame(processed_input_array, columns=cleaned_feature_names)

    # --- NEW ADDITION: Reindex processed_input_data to match model's expected features ---
    # Get the feature names the model expects (from its training phase)
    expected_model_features = model.feature_names_in_

    # Reindex the processed_input_data to exactly match the model's expected features.
    # Fill any missing features (e.g., one-hot encoded categories not present in current input)
    # with 0. This is crucial for consistency.
    processed_input_data = processed_input_data.reindex(columns=expected_model_features, fill_value=0)
    # --- END NEW ADDITION ---

    # Make a churn prediction using the trained model
    prediction = model.predict(processed_input_data).tolist()[0]

    # Return the prediction as a JSON response
    return jsonify({'Subscription': prediction})


# Run the Flask app in debug mode
if __name__ == '__main__':
    super_subscribe_api.run(debug=True)

Overwriting backend_files/app.py


## Dependencies file

In [141]:
%%writefile backend_files/requirements.txt
pandas==2.2.2
numpy==2.0.2
matplotlib==3.10.0
seaborn==0.13.2
joblib==1.4.2
xgboost==2.1.4
requests==2.32.4
huggingface_hub==0.34.0
Flask==3.0.3
Werkzeug==3.0.3
gunicorn==21.2.0
scikit-learn==1.6.1


Overwriting backend_files/requirements.txt


## Docker file

In [142]:
%%writefile backend_files/Dockerfile
FROM python:3.11-slim
WORKDIR /app

COPY requirements.txt .
RUN pip install --upgrade pip setuptools wheel \
 && pip install --no-cache-dir -r requirements.txt

COPY . .

# Install dependencies from the requirements file
#RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Define the command to start the application using Gunicorn
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:super_subscribe_api"]

Overwriting backend_files/Dockerfile


##Deploying backend on huggingface

In [143]:
import os
# for hugging face space authentication to upload files
from huggingface_hub import HfApi, login

access_key = ""  #code to define the token
repo_id = "code"  #code to define the repo id.

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="backend_files",
    repo_id=repo_id,  # Corrected: Pass the repo_id here
    repo_type="space"  # Hugging face repo type "space"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...files/preprocessor.joblib: 100%|##########| 2.43kB / 2.43kB            

  ...ription_gridsearch.joblib: 100%|##########|  490kB /  490kB            

  ...subscription_model.joblib: 100%|##########|  490kB /  490kB            

CommitInfo(commit_url='https://huggingface.co/spaces/dcsamuel/SuperMarketing/commit/7a07a2bb3ffc9d322267798d25fb35f818c2234e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7a07a2bb3ffc9d322267798d25fb35f818c2234e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/dcsamuel/SuperMarketing', endpoint='https://huggingface.co', repo_type='space', repo_id='dcsamuel/SuperMarketing'), pr_revision=None, pr_num=None)

## Deployment frontend

In [144]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

In [158]:
%%writefile frontend_files/app.py

import streamlit as st
import requests

st.set_page_config(layout='wide')

# Create a row for the header and button
col_header_title, col_header_button = st.columns([3, 1]) # 3:1 ratio for title vs button

with col_header_title:
    st.markdown("<h1 style='text-align: left; color: #4CAF50;'>Marketing Optimisation: Customer Subscription Prediction </h1>", unsafe_allow_html=True)

with col_header_button:
    st.markdown("<div style='height: 40px;'></div>", unsafe_allow_html=True) # Spacer for vertical alignment
    predict_button_pressed = st.button("Predict Subscription", type='primary')


# Input fields for the features expected by the backend API
month_options = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
day_of_week_options = ['mon', 'tue', 'wed', 'thu', 'fri']
job_options = ['admin', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
housing_options = ['yes', 'no', 'unknown']
loan_options = ['yes', 'no', 'unknown']
marital_options = ['married', 'single', 'divorced', 'unknown']
education_options = ['primary', 'secondary', 'tertiary', 'unknown']
default_options = ['yes', 'no']
contact_options = ['cellular', 'telephone', 'unknown']
poutcome_options = ['failure', 'other', 'success', 'unknown']

# Create two main columns for a more organized layout of inputs
col1, col2 = st.columns(2)

with col1:
    st.markdown("<h3 style='color: #2196F3;'>Personal & Financial Details 💼</h3>", unsafe_allow_html=True)
    # Nested columns to reduce input field size
    nested_col1_1, nested_col1_2 = st.columns(2)
    with nested_col1_1:
        age = st.number_input("Age", min_value=18, max_value=100, value=30)
        marital = st.selectbox("Marital Status", marital_options)
        default = st.selectbox("Has Credit in Default?", default_options)
        loan = st.selectbox("Has Personal Loan?", loan_options)
    with nested_col1_2:
        job = st.selectbox("Job Type", job_options)
        education = st.selectbox("Education Level", education_options)
        balance = st.number_input("Average Yearly Balance (in Euros)", value=1000)
        housing = st.selectbox("Has Housing Loan?", housing_options)


with col2:
    st.markdown("<h3 style='color: #2196F3;'>Campaign & Contact Information 📞</h3>", unsafe_allow_html=True)
    # Nested columns to reduce input field size
    nested_col2_1, nested_col2_2 = st.columns(2)
    with nested_col2_1:
        contact = st.selectbox("Contact Communication Type", contact_options)
        day_of_week_str = st.selectbox("Day of Week of Last Contact", day_of_week_options)
        campaign = st.number_input("Number of Contacts During Campaign", min_value=1, value=1)
        pdays = st.number_input("Days Since Previous Contact", min_value=-1, value=-1)
    with nested_col2_2:
        month = st.selectbox("Month of Last Contact", month_options)
        duration = st.number_input("Last Contact Duration (seconds)", min_value=0, value=100)
        previous = st.number_input("Previous Campaign Contacts", min_value=0, value=0)
        poutcome = st.selectbox("Outcome of Previous Campaign", poutcome_options)


# Map day of week string to numerical value (1-5 for mon-fri)
day_mapping = {'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5}
day_numerical = day_mapping.get(day_of_week_str, 1) # Default to 1 if not found

# Prepare data for API request (ensure keys match backend's expected keys)
prediction_data = {
    "age": age,
    "job": job,
    "marital": marital,
    "education": education,
    "default": default,
    "balance": balance,
    "housing": housing,
    "loan": loan,
    "contact": contact,
    "day": day_numerical, # Sending numerical day to backend
    "month": month, # Sending as Month to backend
    "duration": duration,
    "campaign": campaign,
    "pdays": pdays,
    "previous": previous,
    "poutcome": poutcome
}

if predict_button_pressed:
    # IMPORTANT: Replace <user_name> and <space_name> with your Hugging Face username and backend space name
    # The repo_id for the backend is dcsamuel/SuperMarketing
    api_endpoint = "endpoint"

    response = requests.post(api_endpoint, json=prediction_data)

    if response.status_code == 200:
        result = response.json()
        prediction = result["Subscription"]

        if prediction == 1:
            st.success("**Prediction: The customer will likely subscribe! 🎉**")
        else:
            st.info("**Prediction: The customer will likely NOT subscribe. 😔**")

    else:
        st.error(f"Error in API request: {response.status_code} - {response.text}")
        st.write("Please ensure the backend API is running and accessible.")

Overwriting frontend_files/app.py


## Dependencies file

In [146]:
%%writefile frontend_files/requirements.txt
requests==2.32.4
streamlit==1.45.0

Overwriting frontend_files/requirements.txt


##Dockerfile

In [147]:
%%writefile frontend_files/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9-slim

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

# Define the command to run the Streamlit app on port 8501 and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

# NOTE: Disable XSRF protection for easier external access in order to make batch predictions

Overwriting frontend_files/Dockerfile


In [159]:
import os
from huggingface_hub import HfApi, login

access_key = "secret"  #define the access token
repo_id = "repo"  #define the repo id

login(token=access_key)


# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="frontend_files",
    repo_id=repo_id,  # Hugging face space id
    repo_type="space"  # Hugging face repo type "space"
)

CommitInfo(commit_url='https://huggingface.co/spaces/dcsamuel/SuperMarketingView/commit/541bc949083031b9e4ecc4c81f3447a1f117e868', commit_message='Upload folder using huggingface_hub', commit_description='', oid='541bc949083031b9e4ecc4c81f3447a1f117e868', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/dcsamuel/SuperMarketingView', endpoint='https://huggingface.co', repo_type='space', repo_id='dcsamuel/SuperMarketingView'), pr_revision=None, pr_num=None)